# MILP Optimization Comparison

Compare all available experiment outputs: ABCROWN, MILP with stable ReLU binaries fixed, and MILP with stable ReLU binaries unfixed.

The notebook reads artifacts generated by recreate.sh or uploaded from fixing-binary-vars-experiment. It does not rerun any benchmark.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 60)

In [ ]:
def find_data_dir():
    candidates = [
        Path.cwd() / 'fixing-binary-vars-experiment',
        Path('/kaggle/input/fixing-binary-vars-experiment'),
        Path('/kaggle/working/fixing-binary-vars-experiment'),
    ]
    for candidate in candidates:
        if candidate.exists() and list(candidate.glob('*.csv')):
            return candidate
    kaggle_root = Path('/kaggle/input')
    if kaggle_root.exists():
        matches = list(kaggle_root.glob('**/fix-3-100-3pixel.csv'))
        if matches:
            return matches[0].parent
    raise FileNotFoundError('Could not find fixing-binary-vars-experiment.')

DATA_DIR = find_data_dir()
print(f'Using data from: {DATA_DIR}')
sorted(path.name for path in DATA_DIR.glob('*'))

In [ ]:
def load_milp_csv(path):
    stem = path.stem
    mode = 'fixed' if stem.startswith('fix-') else 'unfixed'
    experiment = re.sub(r'^(fix|no-fix)-', '', stem)
    csv = pd.read_csv(path)
    csv['optimizer'] = 'MILP'
    csv['configuration'] = f'MILP ({mode} stable binaries)'
    csv['experiment'] = experiment
    csv['source_file'] = path.name
    return csv

def load_crown_csv(path):
    stem = path.stem
    match = re.match(r'crown-(.+)-([^-]+)$', stem)
    experiment = match.group(1) if match else stem.removeprefix('crown-')
    profile = match.group(2) if match else 'unknown'
    csv = pd.read_csv(path)
    csv['optimizer'] = 'ABCROWN'
    csv['configuration'] = f'ABCROWN ({profile})'
    csv['experiment'] = experiment
    csv['source_file'] = path.name
    csv['solver_status'] = csv.get('abcrown_status', csv.get('status'))
    return csv

tables = []
for path in sorted(DATA_DIR.glob('*.csv')):
    if path.name.startswith('crown-'):
        tables.append(load_crown_csv(path))
    elif path.name.startswith(('fix-', 'no-fix-')):
        tables.append(load_milp_csv(path))

results = pd.concat(tables, ignore_index=True)
results['solver_status'] = results.get('solver_status', results['status'])
results[['optimizer', 'configuration', 'experiment', 'status']].head()

In [ ]:
def load_milp_debug():
    rows = []
    for path in sorted(DATA_DIR.glob('*.json')):
        if not path.name.startswith(('fix-', 'no-fix-')):
            continue
        mode = 'fixed' if path.name.startswith('fix-') else 'unfixed'
        experiment = re.sub(r'^(fix|no-fix)-|\.json$', '', path.name)
        for record in json.loads(path.read_text()):
            for direction, details in record.get('directions', {}).items():
                total = details.get('total', {})
                before = details.get('before_presolve', {})
                after = details.get('after_presolve', {})
                progress = details.get('cplex_progress', {})
                rows.append({
                    'optimizer': 'MILP',
                    'configuration': f'MILP ({mode} stable binaries)',
                    'experiment': experiment,
                    'instance_id': record['instance_id'],
                    'direction': direction,
                    'unfixed_before': total.get('unfixed_binary_variables', before.get('unfixed_binary_variables')),
                    'after_binary_variables': after.get('binary_variables', after.get('all_binary_variables')),
                    'presolve_time_sec': after.get('time_sec', after.get('presolve_time_sec')),
                    'nodes_processed': progress.get('nodes_processed'),
                    'nodes_remaining': progress.get('nodes_remaining'),
                    'lp_iterations': progress.get('iterations'),
                })
    return pd.DataFrame(rows)

milp_debug = load_milp_debug()
milp_debug.head()

## Runtime and verification outcome

In [ ]:
summary = (results.groupby(['experiment', 'configuration'])
           .agg(instances=('instance_id', 'nunique'),
                unsat=('status', lambda s: (s == 'unsat').sum()),
                unknown=('status', lambda s: (s == 'unknown').sum()),
                median_runtime_sec=('runtime_sec', 'median'),
                mean_runtime_sec=('runtime_sec', 'mean'),
                max_runtime_sec=('runtime_sec', 'max'))
           .reset_index())
summary.sort_values(['experiment', 'configuration'])

In [ ]:
for region, region_data in [('3pixel', results[results['experiment'].str.contains('3pixel', case=False, na=False)]),
                         ('global', results[results['experiment'].str.contains('global', case=False, na=False)])]:
    if region_data.empty:
        continue
    plt.figure(figsize=(14, 6))
    sns.boxplot(data=region_data, x='experiment', y='runtime_sec', hue='configuration')
    plt.yscale('log')
    plt.ylabel('Runtime (seconds, log scale)')
    plt.xlabel('Network and input region')
    plt.xticks(rotation=30, ha='right')
    plt.title(f'Runtime across optimization configurations: {region}')
    plt.tight_layout()
    plt.show()

## MILP search behavior

In [ ]:
metrics = ['unfixed_before', 'after_binary_variables', 'nodes_processed', 'lp_iterations']
long = milp_debug.melt(
    id_vars=['experiment', 'configuration', 'instance_id', 'direction'],
    value_vars=metrics,
    var_name='metric', value_name='value',
)
long['value'] = pd.to_numeric(long['value'], errors='coerce')
long = long.dropna(subset=['value'])
g = sns.catplot(
    data=long, x='configuration', y='value', col='metric', row='experiment',
    kind='box', sharey=False, height=3, aspect=1.25,
)
g.set_titles('{row_name} | {col_name}')
for ax in g.axes.flat:
    ax.set_yscale('symlog', linthresh=1)
plt.tight_layout()
plt.show()

In [ ]:
numeric_metrics = ['unfixed_before', 'after_binary_variables', 'nodes_processed', 'lp_iterations', 'nodes_remaining']
milp_summary = milp_debug.copy()
milp_summary[numeric_metrics] = milp_summary[numeric_metrics].apply(
    pd.to_numeric, errors='coerce'
)
milp_summary.groupby(['experiment', 'configuration']).agg(
    median_unfixed_before=('unfixed_before', 'median'),
    median_after_binaries=('after_binary_variables', 'median'),
    median_nodes=('nodes_processed', 'median'),
    median_lp_iterations=('lp_iterations', 'median'),
    max_nodes_remaining=('nodes_remaining', 'max'),
).reset_index().sort_values(['experiment', 'configuration'])

## Interpretation

Treat unknown as an incomplete proof or timeout, not as a successful verification. Compare ABCROWN and MILP primarily by completion rate and runtime, then use the MILP debug metrics to explain whether fixing binaries reduced the search space or changed branch-and-bound behavior.